# 📖 Notebook 2: Content Deduplication & Ranking

When a major story breaks, dozens of news outlets publish articles about it within minutes. A good news aggregator must **detect these duplicates** and pick the best version to show. Then it must **rank** all articles so the most important, freshest stories appear first.

## Learning Objectives

By the end of this notebook, you'll understand:
- How exact deduplication works using content hashes (SHA-256)
- How near-duplicate detection works using shingling and Jaccard similarity
- How to group duplicates and pick a canonical article
- How to rank articles by freshness, popularity, and a combined score

## 🛠️ Setup

```bash
cd system-designs/news-aggregator
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import hashlib
import json
import time
import math
from datetime import datetime, timezone

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "newsagg_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db(); conn.close(); print("✅ PostgreSQL")
r = get_redis(); r.ping(); print("✅ Redis")

## 🔍 The Duplicate Problem

Let's look at our seed data. Two pairs of articles report the same story:

1. **GPT-5 release** — covered by TechCrunch and Ars Technica
2. **Lakers championship** — covered by ESPN and BBC

A user shouldn't see both versions. We need to detect these duplicates.

In [ ]:
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Find articles with the same content_hash — these are exact duplicates
cursor.execute("""
    SELECT content_hash, COUNT(*) as count,
           array_agg(title) as titles,
           array_agg(id) as ids
    FROM articles
    GROUP BY content_hash
    HAVING COUNT(*) > 1
    ORDER BY count DESC
""")

dupes = cursor.fetchall()
print(f"🔍 Found {len(dupes)} duplicate clusters by content hash:")
print("=" * 80)
for group in dupes:
    print(f"\n  Hash: {group['content_hash'][:16]}...  ({group['count']} articles)")
    for title, aid in zip(group['titles'], group['ids']):
        print(f"    [{aid:>2}] {title[:65]}")

conn.close()

## 🔑 Strategy 1: Exact Dedup with Content Hashing

The simplest dedup strategy: compute a **hash** of the article's normalised text. If two articles have the same hash, they're duplicates.

```
Article A: "OpenAI Releases GPT-5 With Reasoning Breakthrough"
   → normalise → lowercase, strip punctuation
   → hash → SHA-256 → "a1b2c3d4..."

Article B: "OpenAI Releases GPT-5 With Reasoning Breakthrough"  
   → normalise → same text
   → hash → SHA-256 → "a1b2c3d4..."  ← MATCH!
```

**Limitation**: this only catches *exact* duplicates. "GPT-5 Launches With Major Reasoning Improvements" would get a *different* hash even though it's the same story.

In [ ]:
import re

def compute_content_hash(title: str, summary: str) -> str:
    """
    Create a SHA-256 hash from normalised title + summary.
    Normalisation: lowercase, remove punctuation, collapse whitespace.
    """
    text = f"{title} {summary}".lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)  # keep only letters, numbers, spaces
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return hashlib.sha256(text.encode()).hexdigest()

# Demo: identical articles get the same hash
hash_a = compute_content_hash(
    "Breaking: Major Discovery on Mars",
    "Scientists found water on Mars today."
)
hash_b = compute_content_hash(
    "BREAKING: Major Discovery on Mars!",
    "Scientists found water on Mars today!"
)
hash_c = compute_content_hash(
    "Water Found on Mars in Historic Discovery",
    "Researchers announce the detection of water on the red planet."
)

print("🔑 Content Hash Comparison")
print("=" * 60)
print(f"  Article A: {hash_a[:24]}...")
print(f"  Article B: {hash_b[:24]}...")
print(f"  Article C: {hash_c[:24]}...")
print()
print(f"  A == B? {hash_a == hash_b}  ← same story, minor formatting differences")
print(f"  A == C? {hash_a == hash_c}  ← same story, completely rewritten")
print()
print("💡 Exact hashing catches A≈B but misses A≈C. We need fuzzy matching too.")

## 🧩 Strategy 2: Near-Duplicate Detection with Shingling

To catch articles that report the same story but with different wording, we use **shingling** — a classic information retrieval technique.

### How Shingling Works

1. Break the text into overlapping chunks of *k* consecutive words (called "shingles" or "n-grams")
2. Create a set of these shingles for each article
3. Compare sets using **Jaccard similarity**: `|A ∩ B| / |A ∪ B|`

```
Text: "the cat sat on the mat"
3-shingles: {"the cat sat", "cat sat on", "sat on the", "on the mat"}
```

If two articles share many shingles, they're probably about the same story.

In [ ]:
def make_shingles(text: str, k: int = 3) -> set:
    """
    Break text into overlapping k-word shingles.
    Returns a set of shingle strings.
    """
    # Normalise: lowercase, remove punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text.lower())
    words = text.split()
    if len(words) < k:
        return {" ".join(words)}
    return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}


def jaccard_similarity(set_a: set, set_b: set) -> float:
    """
    Jaccard similarity = |intersection| / |union|
    Returns a value between 0 (no overlap) and 1 (identical).
    """
    if not set_a or not set_b:
        return 0.0
    intersection = set_a & set_b
    union = set_a | set_b
    return len(intersection) / len(union)


# Demo with our GPT-5 articles from the seed data
article_1 = (
    "OpenAI Releases GPT-5 With Reasoning Breakthrough. "
    "OpenAI announced GPT-5 today, featuring a major leap in multi-step reasoning "
    "capabilities. The model can solve complex math and science problems with 95% accuracy."
)

article_2 = (
    "GPT-5 Launches With Major Reasoning Improvements. "
    "OpenAI has released GPT-5. The new model demonstrates significant improvements "
    "in chain-of-thought reasoning, scoring 95% on graduate-level math benchmarks."
)

article_unrelated = (
    "Lakers Win NBA Championship in Game 7 Thriller. "
    "The Los Angeles Lakers defeated the Boston Celtics 108-105 in a dramatic "
    "Game 7 to claim their 18th NBA championship."
)

shingles_1 = make_shingles(article_1)
shingles_2 = make_shingles(article_2)
shingles_3 = make_shingles(article_unrelated)

sim_1_2 = jaccard_similarity(shingles_1, shingles_2)
sim_1_3 = jaccard_similarity(shingles_1, shingles_3)

print("🧩 Shingling + Jaccard Similarity")
print("=" * 55)
print(f"  Article 1 shingles: {len(shingles_1)}")
print(f"  Article 2 shingles: {len(shingles_2)}")
print(f"  Overlap (1∩2):      {len(shingles_1 & shingles_2)}")
print()
print(f"  Similarity (GPT-5 vs GPT-5):    {sim_1_2:.3f}")
print(f"  Similarity (GPT-5 vs Lakers):   {sim_1_3:.3f}")
print()

THRESHOLD = 0.15
print(f"  Threshold: {THRESHOLD}")
print(f"  GPT-5 pair: {'🔴 DUPLICATE' if sim_1_2 >= THRESHOLD else '🟢 Unique'}")
print(f"  GPT-5 vs Lakers: {'🔴 DUPLICATE' if sim_1_3 >= THRESHOLD else '🟢 Unique'}")

In [ ]:
# Let's see the overlapping shingles to understand WHY they matched
common = shingles_1 & shingles_2
print(f"🔍 Shared shingles between the two GPT-5 articles ({len(common)}):")
print("=" * 55)
for shingle in sorted(common):
    print(f"  • \"{shingle}\"")

print()
print("💡 Even though the articles are written differently,")
print("   they share key phrases like 'gpt5', 'reasoning', '95%'.")

## 🏗️ Building Duplicate Groups

When we detect duplicates, we group them together and pick a **canonical** article — the one we'll show in the feed. The others are hidden but their existence boosts the story's importance ("reported by 5 sources").

How to pick the canonical article:
- **Most detailed** — highest word count
- **First published** — the original reporter
- **Most trusted source** — from a tier-1 outlet

Let's implement this against our database.

In [ ]:
def find_near_duplicates(threshold: float = 0.15) -> list:
    """
    Compare all recent articles pairwise using shingling.
    Returns groups of duplicate article IDs.
    
    NOTE: In production, you'd use MinHash/LSH for efficiency.
    This brute-force approach is fine for educational purposes.
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get recent articles (last 48 hours in a real system)
    cursor.execute("""
        SELECT id, title, summary, word_count, published_at
        FROM articles
        ORDER BY published_at DESC
    """)
    articles = cursor.fetchall()
    conn.close()

    # Build shingle sets for each article
    shingle_map = {}
    for a in articles:
        text = f"{a['title']} {a['summary'] or ''}"
        shingle_map[a['id']] = {
            "shingles": make_shingles(text),
            "article": a
        }

    # Compare all pairs
    ids = list(shingle_map.keys())
    pairs = []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            sim = jaccard_similarity(
                shingle_map[ids[i]]["shingles"],
                shingle_map[ids[j]]["shingles"]
            )
            if sim >= threshold:
                pairs.append((ids[i], ids[j], sim))

    # Group connected pairs using union-find
    parent = {aid: aid for aid in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        parent[find(a)] = find(b)

    for a, b, _ in pairs:
        union(a, b)

    groups = {}
    for aid in ids:
        root = find(aid)
        if root not in groups:
            groups[root] = []
        groups[root].append(aid)

    # Only return groups with 2+ articles
    duplicate_groups = [g for g in groups.values() if len(g) > 1]
    return duplicate_groups, shingle_map


dup_groups, shingle_map = find_near_duplicates(threshold=0.15)

print(f"🔍 Found {len(dup_groups)} duplicate groups:")
print("=" * 70)
for i, group in enumerate(dup_groups):
    print(f"\n  Group {i+1}:")
    for aid in group:
        a = shingle_map[aid]["article"]
        print(f"    [{aid:>2}] {a['title'][:55]:<55} ({a['word_count']} words)")

In [ ]:
def pick_canonical(group_ids: list, shingle_map: dict) -> int:
    """
    Pick the best article from a duplicate group.
    Strategy: highest word count (most detailed). Tie-break: earliest published.
    """
    best = None
    for aid in group_ids:
        a = shingle_map[aid]["article"]
        if best is None:
            best = a
        elif a["word_count"] > best["word_count"]:
            best = a
        elif (a["word_count"] == best["word_count"]
              and a["published_at"] and best["published_at"]
              and a["published_at"] < best["published_at"]):
            best = a
    return best["id"]

print("🏆 Canonical article selection:")
print("=" * 70)
for i, group in enumerate(dup_groups):
    canonical_id = pick_canonical(group, shingle_map)
    print(f"\n  Group {i+1} — Canonical: article #{canonical_id}")
    for aid in group:
        a = shingle_map[aid]["article"]
        marker = "  ★ SHOW" if aid == canonical_id else "    hide"
        print(f"  {marker} [{aid:>2}] {a['title'][:50]} ({a['word_count']} words)")

print("\n💡 The user sees one article per story, but we note 'reported by N sources'.")

## 📊 Ranking Articles

After dedup, we need to decide the **order** articles appear in. The three main signals:

| Signal | What It Means | How We Measure It |
|--------|--------------|-------------------|
| **Freshness** | How new is the article? | Exponential time decay |
| **Popularity** | How widely reported / read? | Source count + interaction count |
| **Relevance** | How relevant to this user? | (Covered in Notebook 3) |

### Freshness Score

We use **exponential decay**: an article's freshness drops by half every 6 hours.

```
freshness = e^(-λ × hours_old)

where λ = ln(2) / half_life_hours
```

This means:
- 0 hours old → score 1.0
- 6 hours old → score 0.5
- 12 hours old → score 0.25
- 24 hours old → score 0.06

In [ ]:
HALF_LIFE_HOURS = 6  # freshness halves every 6 hours
DECAY_RATE = math.log(2) / HALF_LIFE_HOURS

def freshness_score(published_at: datetime) -> float:
    """
    Exponential decay based on article age.
    Returns a value between 0 (very old) and 1 (just published).
    """
    if not published_at:
        return 0.0
    now = datetime.now(timezone.utc)
    # Handle timezone-naive datetimes from the database
    if published_at.tzinfo is None:
        published_at = published_at.replace(tzinfo=timezone.utc)
    age_hours = (now - published_at).total_seconds() / 3600
    return math.exp(-DECAY_RATE * max(age_hours, 0))

# Visualise the decay curve
print("📉 Freshness Decay Curve (half-life = 6 hours):")
print("=" * 55)
for hours in [0, 1, 3, 6, 12, 24, 48]:
    score = math.exp(-DECAY_RATE * hours)
    bar = "█" * int(score * 40)
    print(f"  {hours:>3}h old → {score:.3f}  {bar}")

In [ ]:
def popularity_score(source_count: int, interaction_count: int) -> float:
    """
    Score based on how many sources reported the story and user engagement.
    Uses log scale so 100 interactions isn't 100× better than 1.
    """
    # Source count is very valuable — widely reported = important
    source_score = math.log(1 + source_count) / math.log(10)  # normalise to ~0-1
    # Interactions (clicks, reads, shares)
    interaction_score = math.log(1 + interaction_count) / math.log(100)  # normalise
    return 0.6 * source_score + 0.4 * interaction_score


def combined_score(published_at, source_count, interaction_count,
                   freshness_weight=0.6, popularity_weight=0.4) -> float:
    """
    Blend freshness and popularity into a single ranking score.
    """
    f = freshness_score(published_at)
    p = popularity_score(source_count, interaction_count)
    return freshness_weight * f + popularity_weight * p


# Rank all articles in the database
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT a.id, a.title, a.published_at, a.source_count, a.content_hash,
           f.name AS source,
           COUNT(ui.id) AS interactions
    FROM articles a
    JOIN feeds f ON a.feed_id = f.id
    LEFT JOIN user_interactions ui ON a.id = ui.article_id
    GROUP BY a.id, f.name
    ORDER BY a.published_at DESC
""")
articles = cursor.fetchall()
conn.close()

# Score and rank
scored = []
for a in articles:
    f = freshness_score(a["published_at"])
    p = popularity_score(a["source_count"], a["interactions"])
    c = combined_score(a["published_at"], a["source_count"], a["interactions"])
    scored.append({**a, "freshness": f, "popularity": p, "score": c})

scored.sort(key=lambda x: x["score"], reverse=True)

print("📊 Article Rankings (freshness 60% + popularity 40%):")
print("=" * 95)
print(f"  {'#':>2} {'Title':<45} {'Source':<15} {'Fresh':>6} {'Pop':>6} {'Score':>6}")
print("-" * 95)
for i, a in enumerate(scored[:15]):
    print(f"  {i+1:>2} {a['title'][:44]:<45} {a['source'][:14]:<15} "
          f"{a['freshness']:>6.3f} {a['popularity']:>6.3f} {a['score']:>6.3f}")

## 🏆 Ranked Feed with Dedup

Now let's combine deduplication and ranking to produce a clean feed: **one article per story, ranked by combined score**.

In [ ]:
def build_ranked_feed(limit: int = 10) -> list:
    """
    Build a deduplicated, ranked feed:
    1. Fetch all articles
    2. Remove duplicates (keep canonical)
    3. Score and rank
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get articles with their duplicate group info
    cursor.execute("""
        SELECT a.id, a.title, a.published_at, a.source_count, a.summary,
               f.name AS source,
               COUNT(ui.id) AS interactions,
               dg.canonical_article_id
        FROM articles a
        JOIN feeds f ON a.feed_id = f.id
        LEFT JOIN user_interactions ui ON a.id = ui.article_id
        LEFT JOIN duplicate_members dm ON a.id = dm.article_id
        LEFT JOIN duplicate_groups dg ON dm.group_id = dg.id
        GROUP BY a.id, f.name, dg.canonical_article_id
    """)
    articles = cursor.fetchall()
    conn.close()

    # Filter: keep only canonical articles (or those not in any group)
    feed = []
    for a in articles:
        if a["canonical_article_id"] is None:  # not in a dup group
            feed.append(a)
        elif a["id"] == a["canonical_article_id"]:  # is the canonical
            feed.append(a)
        # else: skip (it's a duplicate, not canonical)

    # Score and sort
    for a in feed:
        a["score"] = combined_score(
            a["published_at"], a["source_count"], a["interactions"]
        )

    feed.sort(key=lambda x: x["score"], reverse=True)
    return feed[:limit]


feed = build_ranked_feed(limit=10)

print("📰 Deduplicated & Ranked Feed (Top 10):")
print("=" * 85)
for i, a in enumerate(feed):
    sources = f"({a['source_count']} source{'s' if a['source_count'] > 1 else ''})"
    print(f"\n  {i+1}. {a['title'][:60]}")
    print(f"     {a['source']} {sources} | score: {a['score']:.3f}")
    if a['summary']:
        print(f"     {a['summary'][:80]}...")

## ⚡ Caching Rankings in Redis

Recomputing the ranked feed for every request is expensive. We cache the global ranked feed in a **Redis sorted set** with a TTL, so most reads are instant.

In [ ]:
r = get_redis()
FEED_CACHE_KEY = "global_feed"
FEED_TTL_SECONDS = 300  # 5 minutes

def cache_ranked_feed(feed: list):
    """
    Store the ranked feed in a Redis sorted set.
    Score = ranking score (higher = better).
    Member = JSON blob with article data.
    """
    pipe = r.pipeline()
    pipe.delete(FEED_CACHE_KEY)

    for article in feed:
        member = json.dumps({
            "id": article["id"],
            "title": article["title"],
            "source": article["source"],
            "source_count": article["source_count"],
            "summary": (article["summary"] or "")[:200],
        })
        pipe.zadd(FEED_CACHE_KEY, {member: article["score"]})

    pipe.expire(FEED_CACHE_KEY, FEED_TTL_SECONDS)
    pipe.execute()


def get_cached_feed(offset: int = 0, limit: int = 5) -> list:
    """
    Read the ranked feed from Redis cache (highest score first).
    Returns empty list on cache miss.
    """
    results = r.zrevrange(FEED_CACHE_KEY, offset, offset + limit - 1, withscores=True)
    return [(json.loads(member), score) for member, score in results]


# Cache the feed we just built
cache_ranked_feed(feed)
print(f"✅ Cached {len(feed)} articles in Redis (TTL: {FEED_TTL_SECONDS}s)")

# Read it back — this is what a user request would do
cached = get_cached_feed(offset=0, limit=5)
print(f"\n📰 Reading from cache (page 1, 5 articles):")
print("=" * 70)
for i, (article, score) in enumerate(cached):
    sources = f"({article['source_count']} sources)" if article['source_count'] > 1 else ""
    print(f"  {i+1}. {article['title'][:55]}")
    print(f"     {article['source']} {sources} | score: {score:.3f}")

# Check TTL
ttl = r.ttl(FEED_CACHE_KEY)
print(f"\n⏰ Cache expires in {ttl} seconds")
print("💡 In production, refresh the cache every few minutes with a background job.")

## 🧹 Cleanup

In [ ]:
r = get_redis()
keys = r.keys("global_feed*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

1. **Exact dedup** (content hashing) catches identical articles with minor formatting differences
2. **Near-duplicate detection** (shingling + Jaccard) catches articles about the same story with different wording
3. **Union-find** efficiently groups connected duplicates
4. **Canonical selection** picks the best article from each group to show the user
5. **Exponential decay** ensures fresh news ranks higher than stale news
6. **Log-scale popularity** prevents viral articles from permanently dominating the feed
7. **Redis sorted sets** cache the ranked feed for fast reads

### System Design Interview Tips

- Mention **MinHash/LSH** for production near-duplicate detection — shingling is O(n²), MinHash is O(n)
- Discuss the **dedup threshold** trade-off: too strict = duplicates slip through, too loose = different stories get merged
- Note that **source count** is a powerful signal — a story reported by 10 outlets is more important than one from a single blog

### Next Up

In **Notebook 3**, we'll build **Personalised Feed Generation** — using user interest profiles and reading history to tailor the feed for each individual user.